# Μοντελοποίηση & Εκπαίδευση (Training Phase)

| Step | Task |
|------|------|
| 1    | Διαχωρισμός Benign / Attack και αποθήκευση σε ξεχωριστά αρχεία |
| 2    | 80/20 Split του Benign dataset |
| 3.1  | Κατασκευή Πινάκων Μετάβασης (Transition Matrices) |
| 3.2  | Υπολογισμός Baseline Entropy |
| 3.3  | Επανάληψη για k=1, k=2, k=3 και εξαγωγή αποτελεσμάτων |

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

NOTEBOOK_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = None

for candidate in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
    if (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None and Path("/content/drive/MyDrive").exists():
    # Fast paths to check instead of slow rglob
    candidate_paths = [
        Path("/content/drive/MyDrive/Entropy-Based Network Intrusion Detection using Markov Chains"),
        Path("/content/drive/MyDrive/IT_PROJECT"),
        Path("/content/drive/MyDrive/IT_Project"),
    ]
    for p in candidate_paths:
        if (p / "config.yaml").exists():
            PROJECT_ROOT = p
            break

if PROJECT_ROOT is None:
    PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name in ["2_Data_Pipeline", "3_Training", "4_Detection", "5_Evaluation"] else NOTEBOOK_DIR

os.chdir(PROJECT_ROOT)
print(f"PROJECT_ROOT verified: {PROJECT_ROOT}")
DATA_PATH = PROJECT_ROOT / "data"


In [ ]:
import polars as pl
import numpy as np
import json
import os
from collections import defaultdict
from pathlib import Path

## Step 1 – Διαχωρισμός Benign / Attack
Φορτώνουμε το processed dataset και σπάμε σε δύο ξεχωριστά parquet αρχεία.

In [ ]:
processed_file = DATA_PATH / "processed.parquet"
if not processed_file.exists():
    raise FileNotFoundError(f"File '{processed_file}' not found! Please run 2_Data_Pipeline/data_pipeline.ipynb first.")

df = pl.read_parquet(processed_file)
print(f"Total rows: {df.height}")

df_benign = df.filter(pl.col("Label") == "Benign")
df_attack = df.filter(pl.col("Label") != "Benign")

df_benign.write_parquet(DATA_PATH / "benign.parquet")
df_attack.write_parquet(DATA_PATH / "attack.parquet")

print(f"Benign rows: {df_benign.height}")
print(f"Attack rows: {df_attack.height}")
print(f"Attack types: {df_attack["Label"].unique().to_list()}")


## Step 2 – 80/20 Train/Test Split (Benign)
80% για εκπαίδευση baseline, 20% κρατείται για testing στο Phase 4.

In [ ]:
TRAIN_RATIO = 0.80
n = df_benign.height
split_idx = int(n * TRAIN_RATIO)

df_train = df_benign.head(split_idx)
df_test_benign = df_benign.tail(n - split_idx)

df_train.write_parquet(DATA_PATH / 'benign_train.parquet')
df_test_benign.write_parquet(DATA_PATH / 'benign_test.parquet')

print(f'Train: {df_train.height} rows ({TRAIN_RATIO*100:.0f}%)')
print(f'Test:  {df_test_benign.height} rows ({(1-TRAIN_RATIO)*100:.0f}%)')

## 3.1 – Πίνακες Μετάβασης (Transition Matrices)
Για κάθε discretized feature (`_sym`), κατασκευάζουμε k-th order Markov transition matrix από τα training data.

In [ ]:
SYMBOL_COLS = [
    "Flow_Duration_sym",
    "Out_Degree_sym",
    "Tot_Fwd_Pkts_sym",
    "TotLen_Fwd_Pkts_sym",
    "TCP_Flags_Int_sym"
]

SMOOTHING = 1e-9

def build_transition_matrix(sequence, k=1):
    counts = defaultdict(lambda: defaultdict(int))
    for i in range(len(sequence) - k):
        state = str(tuple(sequence[i:i+k]))
        nxt = str(sequence[i+k])
        counts[state][nxt] += 1
    prob = {}
    for state, nexts in counts.items():
        total = sum(nexts.values())
        prob[state] = {s: c / total for s, c in nexts.items()}
    return prob

print("Transition matrix function ready.")


## 3.2 – Baseline Entropy
Υπολογισμός conditional entropy H(X_n | X_{n-k,...,n-1}) ως baseline κανονικότητας.

In [ ]:
def conditional_entropy(transition_matrix):
    H = 0.0
    for state, nexts in transition_matrix.items():
        probs = np.array(list(nexts.values()))
        H += -np.sum(probs * np.log2(probs + SMOOTHING))
    return H / len(transition_matrix) if len(transition_matrix) > 0 else 0.0

print('Entropy function ready.')

## 3.3 – Εκπαίδευση για k=1, k=2, k=3
Για κάθε feature και κάθε Markov order, κατασκευάζουμε τον πίνακα μετάβασης και υπολογίζουμε το baseline entropy. Τα αποτελέσματα εξάγονται σε τοπικά αρχεία.

In [ ]:
K_ORDERS = [1, 2, 3, 5]

models = {}
baselines = {}
results_summary = []

for feat in SYMBOL_COLS:
    seq = df_train[feat].drop_nulls().to_list()
    print(f'\n--- Feature: {feat} (sequence length: {len(seq)}) ---')
    for k in K_ORDERS:
        tm = build_transition_matrix(seq, k=k)
        h = conditional_entropy(tm)
        key = f'{feat}_k{k}'
        models[key] = tm
        baselines[key] = h
        results_summary.append({'feature': feat, 'k': k, 'states': len(tm), 'baseline_entropy': round(h, 6)})
        print(f'  k={k}  |  states: {len(tm)}  |  baseline H: {h:.6f} bits')

## Εξαγωγή Αποτελεσμάτων
Αποθηκεύουμε τα transition matrices, τα baseline entropy values, και ένα summary σε τοπικά αρχεία.

In [ ]:
models_dir = DATA_PATH.parent / "3_Training" / "models"
models_dir.mkdir(parents=True, exist_ok=True)

with open(models_dir / "transition_matrices.json", "w") as f:
    json.dump(models, f, indent=4)

with open(models_dir / "baselines.json", "w") as f:
    json.dump(baselines, f, indent=4)

summary_df = pl.DataFrame(results_summary)
summary_df.write_csv(models_dir / "training_summary.csv")

print(f"Results exported successfully to {models_dir}")

## Σύνοψη Αποτελεσμάτων

In [ ]:
summary_df = pl.DataFrame(results_summary)
print(summary_df)